# Histogram equalization as one-dimensional Monge transport

This notebook generates `fig:monge-histogram-equalization`.  A grayscale image defines an empirical measure on intensities,
$$
\alpha_N=\frac1N\sum_{k=1}^N\delta_{I_k}\quad\hbox{on }[0,1].
$$
Histogram equalization is the monotone Monge map $T=F_\beta^{-1}\circ F_\alpha$ toward a uniform target law $\beta$.  The displacement interpolation uses
$$
I_{t,k}=(1-t)I_k+tT(I_k).
$$
The exported figure uses the cat photograph from `assets/`: images are arranged on the first row and their intensity histograms on the second row.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps

from figure_style import BLUE, RED, interp_color, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()
ASSETS = ROOT / "notebooks-figures" / "assets"


## Monotone equalization map

For a uniform target, the equalized value of a pixel is its empirical rank among all intensities.  This is exactly the one-dimensional increasing rearrangement.

In [ ]:
def load_gray_cat(n=126):
    img = Image.open(ASSETS / "cat.jpg")
    img = ImageOps.grayscale(img).resize((n, n), Image.Resampling.LANCZOS)
    arr = np.asarray(img, dtype=float) / 255.0
    return arr


def equalize_to_uniform(arr):
    flat = arr.ravel()
    order = np.argsort(flat, kind="mergesort")
    ranks = np.empty_like(flat, dtype=float)
    ranks[order] = (np.arange(flat.size) + 0.5) / flat.size
    return ranks.reshape(arr.shape)

img0 = load_gray_cat()
img_eq = equalize_to_uniform(img0)
times = [0.0, 1/3, 2/3, 1.0]
images = [(1 - t) * img0 + t * img_eq for t in times]


## Exported image and histogram panels

Each time value is exported as two separate PDFs: the image panel and the histogram panel.  LaTeX assembles the two rows and supplies the time labels.

In [ ]:
NAME = "monge-histogram-equalization"
OUT = figure_dir(NAME)

for t, img in zip(times, images):
    suffix = f"t{int(round(100*t)):03d}"
    color = interp_color(t)

    fig, ax = plt.subplots(figsize=(1.38, 1.38))
    ax.imshow(img, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    remove_axes(ax)
    save_pdf(fig, OUT / f"image-{suffix}.pdf", pad_inches=0.01)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(1.38, 0.74))
    bins = np.linspace(0, 1, 42)
    hist, edges = np.histogram(img.ravel(), bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.fill_between(centers, 0, hist, color=color, alpha=0.28, linewidth=0)
    ax.plot(centers, hist, color=color, lw=1.1)
    ax.plot([0, 1], [1, 1], color=BLUE, lw=0.75, alpha=0.85)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(2.2, 1.05 * hist.max()))
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    save_pdf(fig, OUT / f"hist-{suffix}.pdf", pad_inches=0.01)
    plt.close(fig)
